# Chapter 30: backtesting biases

Build a 36-stock, 522-weekday constructed market (seed 20261030) with announcements, splits, dividends and delistings. Weekdays include exchange holidays and are not an actual calendar. The shared ledger trades and marks raw prices and books actions separately. Close-based features use economic returns; lag 1 means next-grid-day open-to-close holdings. Eight studies freshly run constructed variants; they do not replay market results. The French branch is skipped.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch30_backtest_biases_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. Constructed market

Inspect mark/execution price shapes, delistings, the day-125 split, dividend counts and announcement times.

In [ ]:
import pandas as pd
import numpy as np

ch30 = load_code_module("ch30_backtest_biases.py")
data = ch30.build_constructed_market(seed=20261030)

print("Mark prices shape:", data["mark_prices"].shape)
print("Execution prices shape:", data["exec_prices"].shape)
print("Delistings schedule:", data["delistings"])
split_date = str(data["dates"][125].date())
print(f"Split event (S05 on day 125, {split_date}): ratio =", data["splits"].loc[data["dates"][125], "S05"])
print("Dividend payments count: S08 =", (data["dividends"]["S08"] > 0).sum(), ", S12 =", (data["dividends"]["S12"] > 0).sum())
print("Announcement dates count:", len(data["announcement_dates"]))

## 1.1. Research and holding intervals

Close-to-close economic returns can enter a close signal. A lag-1 fill at the next open does not earn the preceding overnight gap. A 16:30 announcement misses that close's information set; its signal first forms at the next grid close and trades one grid day later.

In [ ]:
economic_returns = ch30.economic_daily_returns(data)
open_to_close = ch30.open_to_close_returns(data)
split_date = data["dates"][125]
dividend_date = data["dates"][60]
print("S05 split-day economic return:", economic_returns.loc[split_date, "S05"])
print("S08 dividend-day close-to-close economic return:", economic_returns.loc[dividend_date, "S08"])
print("S08 dividend-day next-open-to-close label:", open_to_close.loc[dividend_date, "S08"])

## 2. Look-ahead bias

Compare future-informed and causal lag-1 routes, retaining Sharpe/NAV differences and unresolved limitations. A constructed difference has no universal sign or magnitude.

In [ ]:
s1 = ch30.study_lookahead(data)

print("=== Bias 1: Look-ahead vs Causal Lag=1 ===")
print(f"Fixed (lag=1) Sharpe:     {s1['fixed']['sharpe']:.4f}, Final NAV: {s1['fixed']['final_nav']:.2f}")
print(f"Wrong (Lookahead) Sharpe: {s1['wrong']['sharpe']:.4f}, Final NAV: {s1['wrong']['final_nav']:.2f}")
print(f"Sharpe Difference:        {s1['diff_sharpe']:.4f} ({s1['sign']})")
print(f"Final NAV Difference:     {s1['diff_final_nav']:.2f} USD")
print("\nUnsolved limitations:", s1["what_fix_leaves_unsolved"])

## 3. Survivorship

Compare ex-post survivors with the dynamic universe and delisting cash policy. Inspect the S35 hand check and the truncated-normal illustration separately.

In [ ]:
s2 = ch30.study_survivorship(data)

print("=== Bias 2: Survivorship Bias vs PIT Universe ===")
print(f"Fixed (PIT + Delist) Sharpe: {s2['fixed']['sharpe']:.4f}, Final NAV: {s2['fixed']['final_nav']:.2f}")
print(f"Wrong (Survivors) Sharpe:    {s2['wrong']['sharpe']:.4f}, Final NAV: {s2['wrong']['final_nav']:.2f}")
print(f"Sharpe Difference:           {s2['diff_sharpe']:.4f} ({s2['sign']})")
print(f"Final NAV Difference:        {s2['diff_final_nav']:.2f} USD")
print("\nS35 Delisting Cash Hand Check:", s2["s35_delisting_hand_check"])
print("Truncated Normal Bias Check:", s2["truncated_normal_check"])
print("\nUnsolved limitations:", s2["what_fix_leaves_unsolved"])

## 4. Selection

Compare full-sample top-ten selection with trailing-60-observation selection. Past-only selection does not by itself establish economic validity.

In [ ]:
s3 = ch30.study_selection(data)

print("=== Bias 3: Ex-post Selection vs Dynamic Trailing Selection ===")
print(f"Fixed (Trailing 60d) Sharpe: {s3['fixed']['sharpe']:.4f}, Final NAV: {s3['fixed']['final_nav']:.2f}")
print(f"Wrong (Ex-post Top10) Sharpe:{s3['wrong']['sharpe']:.4f}, Final NAV: {s3['wrong']['final_nav']:.2f}")
print(f"Sharpe Difference:           {s3['diff_sharpe']:.4f} ({s3['sign']})")
print(f"Final NAV Difference:        {s3['diff_final_nav']:.2f} USD")
print("Top 10 ex-post selected tickers:", s3["top10_expost"])
print("\nUnsolved limitations:", s3["what_fix_leaves_unsolved"])

## 5. Data snooping

Record all 50 attempts and compare the best nominal p-value with multiplicity adjustments and expected-maximum references.

In [ ]:
s4 = ch30.study_data_snooping(data)

print("=== Bias 4: Data Snooping & Multiplicity Adjustments ===")
print(f"Total candidate trials evaluated: {s4['n_trials']}")
print(f"Max candidate Sharpe:             {s4['max_sharpe']:.4f} (lookback={s4['best_trial']['lookback']})")
print(f"Median candidate Sharpe:          {s4['median_sharpe']:.4f}")
print(f"Mean candidate Sharpe:            {s4['mean_sharpe']:.4f}")
print(f"Min candidate Sharpe:             {s4['min_sharpe']:.4f}")
print(f"Exact expected max Z (M=50):      {s4['expected_max_exact']:.4f}")
print(f"BLdP expected max Z (M=50):       {s4['expected_max_bldp']:.4f}")
print(f"Gumbel asymptotic max Z (M=50):   {s4['expected_max_gumbel']:.4f}")
print(f"Best trial nominal p-value:       {s4['p_nominal']:.4f}")
print(f"Bonferroni-adjusted p-value:      {s4['p_bonferroni']:.4f}")
print("\nUnsolved limitations:", s4["what_fix_leaves_unsolved"])

## 6. Overfitting

Compare an eight-lag fitted model with a simple 20-day rule in discovery and holdout. Audit feature and label intervals; the observed decay belongs to this path.

In [ ]:
s5 = ch30.study_overfitting(data)

ov = s5["overfitted_model"]
sm = s5["simple_model"]
print("=== Bias 5: In-Sample Fit vs Out-of-Sample Generalization ===")
print(f"Overfitted 8-Lag Model: IS Sharpe = {ov['is']['sharpe']:.4f} -> OOS Sharpe = {ov['oos']['sharpe']:.4f} (Decay = {ov['sharpe_decay']:.4f})")
print(f"Simple 20-Day Model:    IS Sharpe = {sm['is']['sharpe']:.4f} -> OOS Sharpe = {sm['oos']['sharpe']:.4f} (Decay = {sm['sharpe_decay']:.4f})")
print("Feature interval:", s5["feature_interval"])
print("Label interval:", s5["label_interval"])
print("\nUnsolved limitations:", s5["what_fix_leaves_unsolved"])

## 7. Corporate actions

Compare correct booking with ignored splits and double-counted dividends. Read the hand check and reconciliation residual: adjusted returns and cash booking cannot count a dividend twice.

In [ ]:
s6 = ch30.study_corporate_actions(data)

print("=== Bias 6: Corporate Actions Accounting ===")
print(f"Fixed (Proper Split + Divs) Final NAV: {s6['fixed']['final_nav']:.2f}")
print(f"Split Ignored (Fake -50% Cliff) Final NAV:{s6['split_ignored']['final_nav']:.2f}")
print(f"Dividend Double-Counted Final NAV:     {s6['dividend_double_counted']['final_nav']:.2f}")
print(f"Split day return (Fixed):              {s6['split_day_nav_ret_fixed']:.4f}")
print(f"Split day return (Split Ignored):      {s6['split_day_nav_drop_no_split']:.4f}")
print(f"Reconciliation max residual:           {s6['max_reconcile_residual']:.2e}")
print("Day 125 Split Hand Check:", s6["day_125_split_hand_check"])
print("\nUnsolved limitations:", s6["what_fix_leaves_unsolved"])

## 8. Parameter mining

Inspect the warmup-corrected lookback grid, selected in-sample parameter, its holdout result and the median holdout grid result. Warmup correction does not remove selection bias.

In [ ]:
s7 = ch30.study_parameter_mining(data)

print("=== Bias 7: Parameter Mining Grid Search (Warm-up Corrected) ===")
print(f"Best IS parameter: Lookback L = {s7['best_is_lookback']} days")
print(f"Best IS Sharpe:                  {s7['best_is_sharpe']:.4f}")
print(f"OOS Sharpe of best IS parameter: {s7['best_param_oos_sharpe']:.4f}")
print(f"Median OOS Sharpe across grid:   {s7['median_oos_sharpe']:.4f}")
print(f"Out-of-sample shrinkage:         {s7['oos_shrinkage']:.4f}")

df_grid = pd.DataFrame(s7["grid"])
display(df_grid.round(4))
print("\nUnsolved limitations:", s7["what_fix_leaves_unsolved"])

## 9. Cost modeling

Compare turnover and fee-sensitive NAV, CAGR, Sharpe and fees. Proportional-fee sensitivity is not measured market impact or capacity.

In [ ]:
s8 = ch30.study_cost_modeling(data)

print("=== Bias 8: Transaction Cost Sensitivity ===")
print(f"Annualized turnover: {s8['annualized_turnover']:.2f}%")
cost_df = pd.DataFrame(s8["fee_sensitivity"]).T[["final_nav", "cagr", "sharpe", "total_fees"]]
display(cost_df.round(4))
print("\nUnsolved limitations:", s8["what_fix_leaves_unsolved"])

## 10. French example: skipped

The historical top-three-industry comparison is disabled. It cannot establish PIT availability or actual fills.

In [ ]:
if RUN_FRENCH:
    fr = ch30.study_real_french()
    
    print("=== Real Market Case: Kenneth French 10 Industries ===")
    print(f"Sample range: {fr['first_month']} to {fr['last_month']} ({fr['sample_months']} months)")
    print(f"Industry manifest file SHA-256: {fr['industry_manifest_sha256']}")
    print(f"Industry raw ZIP SHA-256: {fr['industry_raw_zip_sha256']}")
    print(f"Processed SHA-256: {fr['processed_sha256']}")
    print("\nPeriod 1 (1926-1975) Top 3 Industries by Sharpe:")
    for name, sh in fr["period_1"]["top_3_industries"]:
        print(f"  {name}: Sharpe = {sh:.4f}")
    
    p2 = fr["period_2"]
    print(f"\nPeriod 2 (1976-2026) Performance:")
    print(f"  Wrong (Full-Sample Top 3) Sharpe:    {p2['wrong_full_sample_top3_sharpe']:.4f} (CAGR = {p2['wrong_full_sample_top3_cagr']:.4f})")
    print(f"  Fixed (Period 1 Top 3) Sharpe:       {p2['fixed_period1_top3_sharpe']:.4f} (CAGR = {p2['fixed_period1_top3_cagr']:.4f})")
    print(f"  Benchmark (Equal-Weight 10) Sharpe:  {p2['equal_weight_10_sharpe']:.4f} (CAGR = {p2['equal_weight_10_cagr']:.4f})")
    print(f"  Wrong - Fixed Sharpe Difference:     {p2['diff_wrong_minus_fixed_sharpe']:.4f}")
    print(f"  Fixed Excess Decay vs Period 1:      {p2['excess_decay']:.4f}")
    print("\nDisclaimer:", fr["pit_disclaimer"])
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


## 11. Future perturbations

Change future marks, execution prices, announcements, splits, dividends and delisting cash after a cutoff. Check prior targets, fills, shares, weights and NAV. Invariance of the corrected constructed route is a causal boundary check, not a complete production proof.

In [ ]:
pert = ch30.study_future_perturbation(data)

print("=== Future Perturbation Invariance Check ===")
print(f"Cutoff date: {pert['cutoff_date']}")
print(f"Perturbed future inputs: {pert['perturbed_future_inputs']}")
for pipeline in ("fixed_lookahead_diffs_up_to_cutoff", "fixed_selection_diffs_up_to_cutoff"):
    print(pipeline, pert[pipeline])
print("Wrong full-sample selection diffs:", pert["wrong_selection_diffs_up_to_cutoff"])
print("Is fixed pipeline invariant:", pert["is_fixed_invariant"])
print("Is wrong causality violated:", pert["is_wrong_causality_violated"])
assert pert["is_fixed_invariant"] == 1
assert pert["is_wrong_causality_violated"] == 1
print("PASS: future market/action changes leave prior fixed-pipeline outputs unchanged.")